# Pickups

Pickups define relationships between the components of a lens. For instance, in a singlet lens, the radii of curvature can be constrained to be equal in magnitude but opposite in sign.

In [ ]:
import numpy as np

from optiland import optic, optimization
from optiland.optimization import minimize

Define a starting lens:

In [ ]:
lens = optic.Optic()

# add surfaces
lens.surfaces.add(index=0, radius=np.inf, thickness=np.inf)
lens.surfaces.add(index=1, radius=50, thickness=3, material="SK16", is_stop=True)
lens.surfaces.add(index=2, radius=-50, thickness=30)
lens.surfaces.add(index=3)

# set aperture
lens.set_aperture(aperture_type="EPD", value=10)

# set fields
lens.fields.set_type(field_type="angle")
lens.fields.add(y=0)

# set wavelengths
lens.wavelengths.add(value=0.48)
lens.wavelengths.add(value=0.55, is_primary=True)
lens.wavelengths.add(value=0.65)

lens.draw()

Define pickups:

In [ ]:
lens.pickups.add(
    source_surface_idx=1,
    attr_type="radius",
    target_surface_idx=2,
    scale=-1,
    offset=0,
)

Note that pickups can also act on generic surface attributes using the string parameter ttr_type. When doing so, you can use [i] to stand in for the dynamic surface index that is resolved at runtime so the target does not accidentally overwrite the source attribute if you provide a hardcoded index.

As an example, if you wanted to pickup the geometry coefficients, we can provide [i] as the index for the generic path:

`python
lens.pickups.add(
    source_surface_idx=1,
    attr_type='surface_group.surfaces[i].geometry.coefficients',
    target_surface_idx=2
)
`

Note: We omit executing this here since our current lens geometry does not have coefficients.

Define optimization problem:

In [ ]:
problem = optimization.OptimizationProblem()

Add operands (targets for optimization):

In [ ]:
"""
Add a wavefront error operand for all wavelengths.

Use Gaussian quadrature distribution for the rays (see distribution documentation for
more information).
"""

for wave in lens.wavelengths.get_wavelengths():
    input_data = {
        "optic": lens,
        "Hx": 0,
        "Hy": 0,
        "num_rays": 3,
        "wavelength": wave,
        "distribution": "gaussian_quad",
    }
    problem.add_operand(
        operand_type="OPD_difference",
        target=0,
        weight=1,
        input_data=input_data,
    )

Define variables - let first radius of curvature vary (the second surface will match this value, but with opposite sign):

In [ ]:
problem.add_variable(lens, "radius", surface_number=1)

Check initial merit function value and system properties:

In [ ]:
problem.info()

Run optimization:

In [ ]:
result = minimize(problem, "dls")

Print result summary:

In [ ]:
print(result)

Print merit function value and system properties after optimization:

In [ ]:
problem.info()

Draw final lens:

In [ ]:
lens.draw(num_rays=5)

Confirm that the radii of curvature are equal and opposite:

In [ ]:
lens.info()